# Knowledge Graph Pipeline
**Follow-up to `DecisionTreeClassifier.ipynb`**

This notebook continues directly from the trained `clf` and uses the same Ollama stack (`all-minilm` for embeddings, `deepseek-r1:1.5b` for extraction).

**Two-pass strategy:**
- **Pass 1 — Syntactic chunks → Node registry** (what entities exist)
- **Pass 2 — Semantic chunks → Relations + node enrichment** (what connects to what, and how)

**Prerequisites:** run all cells in `DecisionTreeClassifier.ipynb` first so `clf`, `X`, `y`, and `df` are in memory.

## 0. Imports & config

In [ ]:
import json
import re
import pickle
import numpy as np
import requests
from dataclasses import dataclass, field
from typing import Generator

# ── Ollama endpoints (same as notebook 1) ─────────────────────────────────────
OLLAMA_URL    = "http://localhost:11434"
EMBED_MODEL   = "all-minilm"
EXTRACT_MODEL = "deepseek-r1:1.5b"
EMBEDDING_DIM = 384

# ── Pipeline config ───────────────────────────────────────────────────────────
INPUT_FILE       = "2408.13296v3.txt"   # path to your paper
OUTPUT_FILE      = "knowledge_graph.json"
CHUNK_SIZE       = 300    # words per chunk
OVERLAP          = 50     # word overlap between consecutive chunks
DEDUP_THRESHOLD  = 0.92   # cosine similarity to merge duplicate nodes

print("Config ready.")

## 1. Pickle the classifier from notebook 1
Run this cell once to persist `clf` to disk so the pipeline can reload it independently.

In [ ]:
# clf must be in memory from DecisionTreeClassifier.ipynb
with open("decision_tree_clf.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Classifier saved → decision_tree_clf.pkl")

## 2. Data classes

In [ ]:
@dataclass
class Chunk:
    id:    int
    text:  str
    label: str = ""   # filled after classification

@dataclass
class Node:
    id:          int
    name:        str
    type:        str          = "Concept"
    description: str          = ""
    source_chunk: int         = -1
    embedding:   list         = field(default_factory=list, repr=False)

@dataclass
class Relation:
    source:   str
    relation: str
    target:   str
    chunk_id: int

print("Data classes defined.")

## 3. Shared helpers
Same `get_ollama_embedding` pattern as notebook 1 — just reused here.

In [ ]:
def get_ollama_embedding(text: str) -> list:
    """Calls local Ollama embeddings API (identical to notebook 1)."""
    payload = {"model": EMBED_MODEL, "prompt": text}
    try:
        response = requests.post(f"{OLLAMA_URL}/api/embeddings", json=payload, timeout=30)
        response.raise_for_status()
        return response.json()["embedding"]
    except Exception as e:
        print(f"  [embed error] {e}")
        return [0.0] * EMBEDDING_DIM


def manual_features(text: str) -> list:
    """Extract the same 5 manual features used to train clf."""
    tokens             = text.split()
    token_count        = len(tokens)
    special_chars      = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_char_ratio = round(special_chars / max(len(text), 1), 3)
    verbs              = [t for t in tokens if t.lower().endswith(("ing", "ed", "ize", "ises"))]
    verb_density       = round(len(verbs) / max(token_count, 1), 3)
    avg_word_length    = round(sum(len(t) for t in tokens) / max(token_count, 1), 2)
    is_identifier      = 1 if "_" in text and token_count <= 5 else 0
    return [token_count, special_char_ratio, verb_density, avg_word_length, is_identifier]


def build_feature_vector(text: str) -> np.ndarray:
    """Combines manual features + embedding — same pipeline as notebook 1."""
    manual = manual_features(text)
    embed  = get_ollama_embedding(text)
    return np.array(manual + embed).reshape(1, -1)


def cosine_sim(a: list, b: list) -> float:
    a, b  = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0


def _parse_json_response(raw: str) -> list:
    """Robustly extract a JSON array from messy LLM output (handles <think> tags, fences)."""
    raw   = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    raw   = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return []


def _ollama_generate(prompt: str) -> str:
    try:
        r = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={"model": EXTRACT_MODEL, "prompt": prompt, "stream": False},
            timeout=120
        )
        r.raise_for_status()
        return r.json().get("response", "")
    except Exception as e:
        print(f"  [llm error] {e}")
        return "[]"


print("Helpers ready.")

## 4. Stream & classify all chunks
Reads the file once, classifies every chunk with `clf`, then splits into two lists: syntactic and semantic.

In [ ]:
def stream_chunks(path: str, size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> Generator:
    words    = []
    chunk_id = 0
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            words.extend(line.split())
            while len(words) >= size:
                yield Chunk(id=chunk_id, text=" ".join(words[:size]))
                chunk_id += 1
                words = words[size - overlap:]
    if words:
        yield Chunk(id=chunk_id, text=" ".join(words))


syntactic_chunks = []
semantic_chunks  = []

print(f"Streaming and classifying '{INPUT_FILE}' ...")
for chunk in stream_chunks(INPUT_FILE):
    label       = clf.predict(build_feature_vector(chunk.text))[0]
    chunk.label = label
    if label == "Syntactic":
        syntactic_chunks.append(chunk)
    else:
        semantic_chunks.append(chunk)

print(f"Done. {len(syntactic_chunks)} syntactic chunks  |  {len(semantic_chunks)} semantic chunks")

## 5. Pass 1 — Build node registry from syntactic chunks
Each syntactic chunk is treated as a candidate node name. We embed every name and merge near-duplicates.

In [ ]:
NODE_EXTRACT_PROMPT = """The text below is an identifier or technical term extracted from a research paper.
Return ONLY a JSON object with these keys: name, type, description.
Types: Person, Organization, Concept, Method, Dataset, Tool, Location
Keep description to one sentence max. No markdown, no extra text.

Text: {text}
"""

raw_nodes: list[Node] = []

print("Pass 1 — extracting nodes from syntactic chunks ...")
for i, chunk in enumerate(syntactic_chunks):
    response = _ollama_generate(NODE_EXTRACT_PROMPT.format(text=chunk.text))

    # wrap in array so _parse_json_response works uniformly
    clean    = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    clean    = re.sub(r"```(?:json)?", "", clean).strip().rstrip("`")
    # try as object first, then fall back to array parser
    obj      = None
    try:
        obj = json.loads(clean)
    except Exception:
        arr = _parse_json_response(clean)
        obj = arr[0] if arr else None

    if obj and obj.get("name"):
        name  = obj["name"].strip()
        embed = get_ollama_embedding(name)
        raw_nodes.append(Node(
            id           = i,
            name         = name,
            type         = obj.get("type", "Concept"),
            description  = obj.get("description", ""),
            source_chunk = chunk.id,
            embedding    = embed
        ))

print(f"Raw nodes extracted: {len(raw_nodes)}")

### 5a. Deduplicate nodes

In [ ]:
def deduplicate_nodes(nodes: list, threshold: float = DEDUP_THRESHOLD):
    """
    Merge nodes whose name embeddings exceed `threshold` cosine similarity.
    Returns (canonical_nodes, alias_map).
    alias_map[merged_name] = canonical_name
    """
    canonical  = []
    alias_map  = {}

    for node in nodes:
        merged = False
        for canon in canonical:
            if cosine_sim(node.embedding, canon.embedding) >= threshold:
                # keep the shorter / cleaner name as canonical
                if len(node.name) < len(canon.name):
                    alias_map[canon.name] = node.name
                    canon.name = node.name
                    # prefer the richer description
                    if len(node.description) > len(canon.description):
                        canon.description = node.description
                else:
                    alias_map[node.name] = canon.name
                merged = True
                break
        if not merged:
            canonical.append(node)

    return canonical, alias_map


node_registry, alias_map = deduplicate_nodes(raw_nodes)

# Re-index IDs after dedup
for idx, node in enumerate(node_registry):
    node.id = idx

# Quick preview
import pandas as pd
registry_df = pd.DataFrame([
    {"id": n.id, "name": n.name, "type": n.type, "description": n.description[:80]}
    for n in node_registry
])
print(f"Node registry: {len(node_registry)} canonical nodes  (merged {len(raw_nodes) - len(node_registry)} duplicates)")
registry_df.head(10)

## 6. Pass 2 — Extract relations from semantic chunks
For each semantic chunk the LLM is given the list of **known node names** and asked to find relations *only between those nodes*. This anchors extraction to the registry and prevents hallucinated entity names.

In [ ]:
RELATION_PROMPT = """You are a knowledge graph builder.
Known entities: {known_entities}

Read the text below and extract relationships ONLY between the known entities above.
Return ONLY a JSON array. Each item: {{"source": "...", "relation": "...", "target": "..."}}
Use short verb phrases for relation (e.g. "improves", "is part of", "evaluates", "proposes").
If no relationships are found, return []. No markdown, no explanation.

Text:
{text}
"""

# Build a compact name list for the prompt (avoid token bloat)
known_names = [n.name for n in node_registry]
known_str   = ", ".join(known_names)

all_relations: list[Relation] = []
node_enrichments: dict        = {}   # name → list of descriptions found in semantic chunks

print(f"Pass 2 — extracting relations from {len(semantic_chunks)} semantic chunks ...")
for chunk in semantic_chunks:

    # ── find which known nodes are mentioned in this chunk ────────────────────
    mentioned = [name for name in known_names if name.lower() in chunk.text.lower()]
    if len(mentioned) < 2:
        continue   # need at least 2 nodes to form a relation

    # ── relation extraction ───────────────────────────────────────────────────
    prompt    = RELATION_PROMPT.format(known_entities=", ".join(mentioned), text=chunk.text)
    raw       = _ollama_generate(prompt)
    relations = _parse_json_response(raw)

    for item in relations:
        src = alias_map.get(item.get("source", "").strip(), item.get("source", "").strip())
        tgt = alias_map.get(item.get("target", "").strip(), item.get("target", "").strip())
        rel = item.get("relation", "").strip()
        # only keep relations between known nodes
        if src in known_names and tgt in known_names and rel:
            all_relations.append(Relation(source=src, relation=rel, target=tgt, chunk_id=chunk.id))

    # ── enrich node descriptions from semantic context ────────────────────────
    for name in mentioned:
        node_enrichments.setdefault(name, [])
        # grab the sentence(s) in the chunk that mention the node
        sentences = [s.strip() for s in re.split(r'(?<=[.!?]) +', chunk.text) if name.lower() in s.lower()]
        node_enrichments[name].extend(sentences[:2])   # cap at 2 sentences per chunk

print(f"Relations extracted: {len(all_relations)}")

### 6a. Enrich node descriptions with semantic context

In [ ]:
for node in node_registry:
    extra = node_enrichments.get(node.name, [])
    if extra:
        # append up to 3 unique supporting sentences to the node description
        seen   = set()
        unique = []
        for s in extra:
            if s not in seen:
                seen.add(s)
                unique.append(s)
            if len(unique) == 3:
                break
        node.description = (node.description + " " + " ".join(unique)).strip()

print("Node descriptions enriched from semantic context.")

## 7. Inspect the graph

In [ ]:
relations_df = pd.DataFrame([
    {"source": r.source, "relation": r.relation, "target": r.target, "chunk_id": r.chunk_id}
    for r in all_relations
])

print(f"Nodes : {len(node_registry)}")
print(f"Edges : {len(all_relations)}")
print()
relations_df.head(15)

In [ ]:
# Most connected nodes
from collections import Counter

all_mentions = [r.source for r in all_relations] + [r.target for r in all_relations]
degree       = Counter(all_mentions)

degree_df = pd.DataFrame(degree.most_common(20), columns=["node", "degree"])
print("Top 20 most connected nodes:")
degree_df

## 8. Export to JSON

In [ ]:
graph = {
    "nodes": [
        {
            "id":          n.id,
            "name":        n.name,
            "type":        n.type,
            "description": n.description
        }
        for n in node_registry
    ],
    "edges": [
        {
            "source":   r.source,
            "relation": r.relation,
            "target":   r.target,
            "chunk_id": r.chunk_id
        }
        for r in all_relations
    ]
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(graph, f, indent=2, ensure_ascii=False)

print(f"Graph saved → {OUTPUT_FILE}")
print(f"  {len(graph['nodes'])} nodes")
print(f"  {len(graph['edges'])} edges")

## 9. (Optional) Visualise with NetworkX

In [ ]:
# pip install networkx matplotlib  (if not already installed)
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()

for n in node_registry:
    G.add_node(n.name, type=n.type)

for r in all_relations:
    G.add_edge(r.source, r.target, label=r.relation)

# Draw top-30 nodes by degree for readability
top_nodes = [node for node, _ in degree.most_common(30)]
subG      = G.subgraph(top_nodes)

plt.figure(figsize=(14, 10))
pos    = nx.spring_layout(subG, seed=42, k=2)
labels = nx.get_edge_attributes(subG, "label")

nx.draw_networkx_nodes(subG,  pos, node_size=800, node_color="#AFA9EC", alpha=0.9)
nx.draw_networkx_labels(subG, pos, font_size=8,   font_color="#26215C", font_weight="bold")
nx.draw_networkx_edges(subG,  pos, edge_color="#888780", arrows=True, arrowsize=15, alpha=0.7)
nx.draw_networkx_edge_labels(subG, pos, edge_labels=labels, font_size=6, font_color="#5F5E5A")

plt.title("Knowledge Graph — top 30 nodes by degree", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.savefig("knowledge_graph.png", dpi=150)
plt.show()
print("Graph visualisation saved → knowledge_graph.png")